In [17]:
# 1-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_1_1.csv')
# print(df.info())
# print(df.head())
df=df[(df['ProductA가격']!=0) & (df['ProductB가격']!=0)]
df['a']=np.abs(df['ProductA가격']-df['ProductB가격'])
print(np.round(df.groupby('도시명')['a'].mean().max(),1))

16333.3


In [23]:
# 1-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_1_2.csv')
# print(df.info())
# print(df.head())

df['BMI']=df['Weight_kg']/(df['Height_cm']*0.01)**2
# print(df.head())
df[(df['BMI'] < 18.5) | (df['BMI'] >= 25)].count()

Height_cm    74
Weight_kg    74
BMI          74
dtype: int64

In [68]:
# 1-3
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_1_3.csv')
# print(df.info())
# print(df.head())

df['순생산량']=(df['products_made_domestic']+df['products_made_international'])-(df['products_sold_domestic']+df['products_sold_international'])
df1=df.groupby(['year', 'factory'])['순생산량'].sum().reset_index()
df1
df2=df1.groupby('year')['순생산량'].idxmax()
df2
df3=df1.loc[df2]
df3['순생산량'].sum()

np.int64(9130)

In [107]:
# 2
import numpy as np
import pandas as pd

train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_2_test.csv')
# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

# train_df=train_df.dropna()
# test_df=test_df.dropna()
# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

train_y=train_df['DBP']
train_X=train_df.drop(columns=['level_0', 'ID', 'DBP'])
test_y=test_df['DBP']
test_X=test_df.drop(columns=['level_0', 'ID', 'DBP'])
# print(train_y, train_X, test_y, test_X)

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import get_scorer_names, root_mean_squared_error
from sklearn.impute import SimpleImputer
# root_mean_squared_error

train_X, valid_X, train_y, valid_y=train_test_split(train_X, train_y, test_size=0.3)
# print(train_X.shape, train_y.shape, valid_X.shape, valid_y.shape)

obj=train_X.select_dtypes('object').columns.tolist()
num=train_X.select_dtypes('number').columns.tolist()

encoder=OneHotEncoder(sparse_output=False, handle_unknown='ignore')
imputer=SimpleImputer(strategy='most_frequent')
train_X_encoder=encoder.fit_transform(train_X[obj])
valid_X_encoder=encoder.transform(valid_X[obj])
test_X_encoder=encoder.transform(test_X[obj])
train_X_impute=imputer.fit_transform(train_X[num])
valid_X_impute=imputer.transform(valid_X[num])
test_X_impute=imputer.transform(test_X[num])

train_X=np.concatenate([train_X_encoder, train_X_impute], axis=1)
valid_X=np.concatenate([valid_X_encoder, valid_X_impute], axis=1)
test_X=np.concatenate([test_X_encoder, test_X_impute], axis=1)

rf=RandomForestRegressor()
rf.fit(train_X, train_y)
pred1=rf.predict(valid_X)
print(valid_y.mean())
print(root_mean_squared_error(valid_y, pred1))

params={
    'max_depth': [10,20,30],
    'min_samples_split': [2,5,10]
}
cv=GridSearchCV(estimator=rf, param_grid=params, scoring='neg_root_mean_squared_error', cv=3)
cv.fit(train_X, train_y)
pred2=cv.predict(valid_X)
print(root_mean_squared_error(valid_y, pred2))

pred3=rf.predict(test_X)
pred4=cv.predict(test_X)

pd.DataFrame(pred4, columns=['pred']).to_csv('result.csv', index=False)

75.41758241758242
10.7854006431364
10.833707947361466


In [78]:
# 3-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_3_1.csv')
# print(df.info())
# print(df.head())

print(np.round(df['work_hours'].mean(), 3))
print(np.round(df['work_hours'].std(), 3))
from scipy.stats import kstest
mu=df['work_hours'].mean()
sigma=df['work_hours'].std()
print(np.round(kstest(df['work_hours'], 'norm', args=(mu, sigma)).statistic,3))
print(kstest(df['work_hours'], 'norm', args=(mu, sigma)).pvalue < 0.05)

8.09
1.519
0.064
False


In [89]:
# 3-2
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_3_2.csv')
# print(df.info())
# print(df.head())
model=smf.ols('price ~ area + rooms + age', data=df).fit()
# print(model.summary())
print(model.params[1:].idxmax())
# model.params
(model.pvalues[1:] < 0.05).sum()

rooms


np.int64(3)